# 07 · RAG basics: chunks, embeddings and semantic search

**Goal:** answer questions about a document by giving Claude only the parts that matter. You build every piece of the pipeline yourself, so nothing is hidden in a library.

| | |
|---|---|
| **You will learn** | Why RAG exists, three ways to cut a document into chunks, what embeddings are, cosine similarity, a small vector index, the full retrieve-then-answer flow, and where semantic search fails |
| **Prerequisites** | [Notebook 01](01_conversations_and_control.ipynb). You also need a free [Voyage AI](https://www.voyageai.com/) API key |
| **Run time** | About 2 minutes |
| **Cost** | Under $0.05 expected for Claude. Voyage embeddings use its free allowance. This notebook stops itself at $0.15 |
| **Paid cells** | Claude: sections 6 and 7. Voyage: sections 3 and 5, and only on the first run, because results are saved |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## What is RAG?

Think of an open-book exam with a 2,000-page textbook. You can't read every page for every question. You check the index, open the two right pages, and answer from those.

**RAG (Retrieval Augmented Generation)** does the same: **retrieve** the few relevant pieces of a document, then let Claude **generate** an answer from them.

```
 question ──▶ find the relevant chunks ──▶ prompt = instructions + chunks + question ──▶ Claude ──▶ answer
```

**Our document** is `data/halvora_report.md`, the annual report of a fictional company called Halvora Systems. It has a few traps built in, which later sections use:

- **Section 1 (Medical Research)** calls a pathogen a *"bug"*, while **Section 2 (Software Engineering)** counts the bugs engineers fixed.
- The incident ID **`INC-2023-Q4-011`** appears in only three sections: 2, 10 and 12.

This notebook builds the basic pipeline. [Notebook 08](08_rag_advanced.ipynb) makes the search smarter.

## Setup

Two clients this time: Claude, and Voyage AI for embeddings. Anthropic doesn't make an embedding model, so the course uses Voyage. Add `VOYAGE_API_KEY` to your `.env` file. Run the cells from top to bottom.

In [ ]:
import hashlib
import json
import math
import os
import re
from pathlib import Path

import voyageai
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

voyage_key = os.getenv("VOYAGE_API_KEY", "")
assert voyage_key and voyage_key != "your_voyage_key_here", "Add your VOYAGE_API_KEY to .env, restart the kernel, and run this again."

client = Anthropic()             # finds ANTHROPIC_API_KEY by itself
voyage_client = voyageai.Client()  # finds VOYAGE_API_KEY by itself
MODEL = "claude-sonnet-5-5"
EMBED_MODEL = "voyage-4-lite"

print("Ready. Claude model:", MODEL, "| Embedding model:", EMBED_MODEL)

In [ ]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.15     # the most this notebook should spend on Claude

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""

## 1. Why not paste in the whole document? (free)

There are two ways to answer a question about a document:

| | Option 1: the whole document in the prompt | Option 2: RAG, only the relevant chunks |
|---|---|---|
| Setup | None | Cut the document up in advance, and build a search |
| Prompt size | The whole document, for **every** question | A few chunks |
| Limits | Very large documents don't fit | Works for huge documents, and many of them |
| Focus | Claude can lose focus in a very long prompt | Claude sees only what matters |
| Cost | You pay for every token, every time | Smaller prompts are cheaper and faster |
| Risk | None | The search can miss the right chunk |

The free `count_tokens` endpoint shows the size difference for one question.

In [ ]:
report_text = Path("data/halvora_report.md").read_text(encoding="utf-8")
headers = [line for line in report_text.splitlines() if line.startswith("## ")]

print(f"Characters: {len(report_text):,} | Words: {len(report_text.split()):,} | '## ' headers: {len(headers)}")
for header in headers:
    print("   ", header)

question = "What did the software engineering team do this year?"


def count_prompt_tokens(context):
    prompt = f"<report>\n{context}\n</report>\n\nAnswer this question about the report: {question}"
    return client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": prompt}]).input_tokens


# Option 2 by hand: we cut out Section 2 ourselves, because we already know where the answer is.
section_2 = "## Section 2:" + report_text.split("## Section 2:")[1].split("\n## ")[0]

full_tokens = count_prompt_tokens(report_text)
section_tokens = count_prompt_tokens(section_2)
print(f"\nOption 1, the whole report: {full_tokens:>6,} input tokens")
print(f"Option 2, only Section 2  : {section_tokens:>6,} input tokens")
print(f"Option 2 sends {full_tokens / section_tokens:.1f}x fewer tokens for the same question.")

assert section_tokens < full_tokens

**What to notice**

- That ratio is the whole argument for RAG, and you pay it on **every** question.
- **To be honest:** this report is small, a few thousand tokens, and fits easily in Claude's context window. For a document this size, Option 1 is fine. RAG pays off when documents are **big**, **many**, or asked about **often**.
- **The catch:** we cut out Section 2 because we knew where the answer was. A real system doesn't know that. The rest of this notebook does that cut **automatically**: first by chunking (section 2), then by searching (sections 3 to 5).

## 2. Chunking: how to cut the document (free)

Think of cutting a newspaper into clippings for a file:

1. **By size:** cut every 10 cm, whatever is there. Fast, but it cuts through words and stories.
2. **By sentence:** cut only at the end of a sentence. Cleaner, but a story can still be split.
3. **By structure:** cut out each article by its headline. Cleanest, but only works when the paper has headlines.
4. **By meaning (semantic):** group paragraphs about the same story. Often the best, but slow and costly because a model reads everything. We won't build this one.

Whatever you cut is exactly what gets retrieved and put in Claude's prompt. A bad cut means half the context, or two topics mixed together.

In [ ]:
def chunk_by_char(text, chunk_size=500, chunk_overlap=50):
    """Fixed-size chunks. Each chunk repeats the last chunk_overlap characters of the one before."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start = end - chunk_overlap if end < len(text) else len(text)
    return chunks


def chunk_by_sentence(text, max_sentences=5, overlap_sentences=1):
    """Groups of sentences. A sentence ends at . ! or ? followed by whitespace."""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start = 0
    while start < len(sentences):
        chunks.append(" ".join(sentences[start:start + max_sentences]))
        start += max_sentences - overlap_sentences
    return chunks


def chunk_by_section(text):
    """One chunk per markdown section. The look-ahead (?=## ) splits before each header and keeps it."""
    return re.split(r"\n(?=## )", text)


char_chunks = chunk_by_char(report_text)
sentence_chunks = chunk_by_sentence(report_text)
section_chunks = chunk_by_section(report_text)

print(f"--- end of size-based chunk 10 ---\n...{char_chunks[10][-120:]}")
print(f"\n--- start of size-based chunk 11 ---\n{char_chunks[11][:120]}...")

**What to notice**

- Size-based chunks start and end wherever character 500 falls, often in the middle of a word.
- The **overlap** repeats the last 50 characters of one chunk at the start of the next, so a sentence cut at the edge still appears whole in at least one chunk.

> **Different from the course:** the course splits on `"\n## "`, and that removes the `## ` from every header. The look-ahead `(?=## )` keeps it, and the header is one of the strongest clues for search later.

Let's compare the three strategies with two measurements: chunks that **start mid-word**, and chunks that **mix two sections**.

In [ ]:
def mixed_topic_count(chunks):
    """Chunks with a '## ' header that isn't at the very start, so they hold parts of two sections."""
    return sum(1 for chunk in chunks if "## " in chunk.lstrip()[1:])


def mid_word_count(chunks, text):
    """Chunks that start in the middle of a word in the original text."""
    count = 0
    for chunk in chunks:
        position = text.find(chunk[:40])
        if position > 0 and text[position - 1].isalnum() and text[position].isalnum():
            count += 1
    return count


print(f"{'strategy':<17}{'chunks':>7}{'avg chars':>11}{'min':>6}{'max':>7}{'mid-word':>10}{'mixed':>7}")
for name, chunks in [("size (500/50)", char_chunks), ("sentence (5/1)", sentence_chunks), ("section (##)", section_chunks)]:
    lengths = [len(chunk) for chunk in chunks]
    print(f"{name:<17}{len(chunks):>7}{sum(lengths) // len(lengths):>11}{min(lengths):>6}{max(lengths):>7}"
          f"{mid_word_count(chunks, report_text):>10}{mixed_topic_count(chunks):>7}")

print("\nSection chunks:")
for number, chunk in enumerate(section_chunks):
    print(f"   {number:>2}. {chunk.splitlines()[0][:55]:<55} ({len(chunk):,} chars)")

assert mixed_topic_count(section_chunks) == 0
assert len(section_chunks) == 16

**What to notice**

- Section chunks score zero on both measures, but only because this report has clean `## ` headers. A scanned PDF or a plain email has none, so this strategy can't be used there.
- Chunk 2 is the **Table of Contents**. It mentions every topic, so it can look relevant to many questions. Menus, indexes and navigation pages cause the same problem in real systems.

**The "bug" trap.** Plain word matching can't tell meanings apart:

In [ ]:
print("Section chunks containing the word 'bug' or 'bugs':")
for chunk in section_chunks:
    hits = len(re.findall(r"\bbugs?\b", chunk, flags=re.IGNORECASE))
    if hits:
        print(f"   {hits} x 'bug'  ->  {chunk.splitlines()[0]}")

Two sections match, and the medical one uses "bug" for a pathogen. A question like *"How many bugs did engineers fix?"* needs search that compares **meaning**, not letters. That's what embeddings do.

| Strategy | Use it when | Watch out for |
|---|---|---|
| **Size + overlap** | Any content: PDFs, code, emails. A common default | Cut words and mixed topics. Overlap helps |
| **Sentence** | Normal prose | Headers glued to sentences, uneven sizes |
| **Structure** | You control the format, such as internal wikis | Fails on documents without structure |
| **Semantic** | Quality matters more than cost | A model has to read everything |

From here on we use the **section chunks**, because this report is clean markdown.

## 3. Embeddings: turning meaning into numbers (Voyage, saved after the first run)

Think of GPS coordinates. Places close together on the map get similar coordinates. An **embedding** gives a piece of text a "coordinate" in 1,024 numbers instead of 2, and texts with similar **meaning** land close together, even when they share no words.

One difference from GPS: nobody knows what each of the 1,024 numbers means. The embedding model learned them in training.

| Setting | Our choice | Why |
|---|---|---|
| Model | `voyage-4-lite` | Voyage's cheapest current model: $0.02 per million tokens, after 200 million free tokens |
| `input_type` | `"document"` for chunks, `"query"` for questions | Voyage adds a different hidden instruction to each. The course uses `"query"` for both, which is a small mistake |
| Batching | Many texts in one call | Up to 1,000 texts per request. Fewer requests also helps with rate limits |
| Disk cache | `data/embeddings_cache.json` | Re-running costs nothing. The repo includes this file, so if you cloned it, most embeddings are already there |

> **Rate limits:** Voyage accounts without a payment method may have much lower rate limits. This notebook batches its calls, so it makes only a few requests. If you see a `RateLimitError`, wait a minute and re-run the cell.

In [ ]:
CACHE_PATH = Path("data/embeddings_cache.json")
embedding_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}


def cache_key(text, input_type):
    # The same model, input_type and text always give the same vector, so we never pay twice.
    return hashlib.sha256(f"{EMBED_MODEL}|{input_type}|{text}".encode("utf-8")).hexdigest()


def generate_embedding(text, input_type):
    """Embed one string or a list of strings. input_type is "document" for chunks, "query" for questions."""
    texts = [text] if isinstance(text, str) else list(text)
    missing = list(dict.fromkeys(t for t in texts if cache_key(t, input_type) not in embedding_cache))

    if missing:  # one batched call for everything not cached yet
        result = voyage_client.embed(missing, model=EMBED_MODEL, input_type=input_type)
        for t, vector in zip(missing, result.embeddings):
            embedding_cache[cache_key(t, input_type)] = vector
        CACHE_PATH.write_text(json.dumps(embedding_cache))
        print(f"   (Voyage call: {len(missing)} text(s), {result.total_tokens:,} tokens)")

    vectors = [embedding_cache[cache_key(t, input_type)] for t in texts]
    return vectors[0] if isinstance(text, str) else vectors


section_embeddings = generate_embedding(section_chunks, input_type="document")

vector = section_embeddings[4]
print(f"\n{len(section_chunks)} chunks -> {len(section_embeddings)} vectors")
print("Chunk 4 starts with:", section_chunks[4].splitlines()[0])
print("Numbers in its vector:", len(vector))
print("First 6 numbers     :", [round(x, 4) for x in vector[:6]])
print("Vector length (norm):", round(math.sqrt(sum(x * x for x in vector)), 4))

assert len(vector) == 1024

**What to notice**

- Every chunk, short or long, gets a vector of the same length: 1,024 numbers.
- If no `(Voyage call: ...)` line was printed, everything came from the cache, and that cell cost nothing.
- **The norm** is the vector's length. Voyage's docs don't say whether its vectors are normalised, so we check. If it prints 1.0, that matters in section 4.

**Same words, different `input_type`:**

In [ ]:
as_query = generate_embedding(question, input_type="query")
as_document = generate_embedding(question, input_type="document")

print("as 'query'   :", [round(x, 4) for x in as_query[:5]])
print("as 'document':", [round(x, 4) for x in as_document[:5]])
print("Identical?   :", as_query == as_document)

The same sentence gets different vectors. Voyage puts *"Represent the query for retrieving supporting documents:"* before a query, and *"Represent the document for retrieval:"* before a document. Questions are written very differently from the passages that answer them, and these instructions help place one near the other.

**Rule:** embed stored chunks as `"document"`, and the user's question as `"query"`.

## 4. Measuring "nearby": cosine similarity (free)

Think of two people pointing at the sky. If they point at the same star, their arms point the **same direction**, even if one arm is longer. **Cosine similarity** measures exactly that: the angle between two vectors, ignoring their length.

$$\text{cosine}(a, b) = \frac{a \cdot b}{|a| \times |b|}$$

- **`a · b`** (the dot product): multiply matching numbers and add them up.
- **`|a|`**: the vector's length, its norm.
- The score runs from **-1 to +1**: about **1** means the same meaning, about **0** means unrelated.

In [ ]:
def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))


def vector_norm(v):
    return math.sqrt(dot_product(v, v))


def cosine_similarity(a, b):
    return dot_product(a, b) / (vector_norm(a) * vector_norm(b))


print("Two-number arrows first, so you can picture them:")
examples = {
    "same direction, longer arrow": ([1, 0], [3, 0]),
    "45 degrees apart": ([1, 0], [1, 1]),
    "90 degrees (unrelated)": ([1, 0], [0, 1]),
    "opposite directions": ([1, 0], [-1, 0]),
}
for label, (a, b) in examples.items():
    print(f"   {label:<30} {str(a):>8} vs {str(b):<8} -> {cosine_similarity(a, b):+.3f}")


def chunk_title(chunk):
    return chunk.splitlines()[0]


titles = [chunk_title(chunk) for chunk in section_chunks]
medical = section_embeddings[titles.index("## Section 1: Medical Research")]
software = section_embeddings[titles.index("## Section 2: Software Engineering")]

print("\nReal embeddings, Medical vs Software sections:")
print(f"   cosine similarity: {cosine_similarity(medical, software):.4f}")
print(f"   dot product      : {dot_product(medical, software):.4f}")

assert abs(cosine_similarity([1, 0], [3, 0]) - 1) < 1e-9

**What to notice**

- `[1, 0]` and `[3, 0]` score +1.000: same direction, and length doesn't matter. That's why cosine is used for meaning.
- **If the two numbers for the real sections match,** Voyage's vectors have length 1. Then cosine similarity is just the dot product, a shortcut many vector databases rely on. We keep the full formula, so the code also works with models that don't normalise.

## 5. Semantic search and a vector index (Voyage, saved after the first run)

Now we score every section against a question. We embed **all** the questions this notebook uses in **one** batched call, which keeps the number of Voyage requests low.

In [ ]:
QUESTIONS = {
    "bugs": "How many bugs did engineers fix this year?",
    "engineering": "What did the software engineering team do this year?",
    "revenue": "How did revenue and profit change in 2023?",
    "hiring": "How many people did the company hire, and where did it open an office?",
    "incident": "What happened with INC-2023-Q4-011?",
}
question_vectors = dict(zip(QUESTIONS, generate_embedding(list(QUESTIONS.values()), input_type="query")))

ranked = sorted(
    ((cosine_similarity(question_vectors["bugs"], vector), chunk_title(chunk))
     for vector, chunk in zip(section_embeddings, section_chunks)),
    reverse=True,
)

print(f"Question: {QUESTIONS['bugs']}\n")
for rank, (score, title) in enumerate(ranked, start=1):
    print(f"   {rank:>2}. {score:.3f} {'#' * max(0, round(score * 40)):<24} {title[:45]}")

**What to notice**

- Word matching in section 2 found **two** sections containing "bug". Here, Section 2 (Software Engineering) should rank well above Section 1 (Medical), even though Section 1 literally contains the word. The embedding compares **meaning**: engineers fixing bugs.
- Look at the **gap** between #1 and #2. A big gap means the search is confident.

Now we package the search as a small class. A **vector index** keeps each vector next to the text it came from, and searches them by similarity.

In [ ]:
class VectorIndex:
    def __init__(self):
        self.vectors = []     # one embedding per document
        self.documents = []   # the text each embedding came from, at the same position

    def add_vectors(self, vectors, documents):
        for vector, document in zip(vectors, documents):
            self.vectors.append(vector)
            self.documents.append(document)

    def search(self, query_vector, k=3):
        """Return the k most similar documents as (score, document), best first."""
        scored = [(cosine_similarity(query_vector, vector), document)
                  for vector, document in zip(self.vectors, self.documents)]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return scored[:k]

    def __len__(self):
        return len(self.documents)


index = VectorIndex()
index.add_vectors(section_embeddings, section_chunks)   # vectors we already have: no new Voyage call

for key in ["engineering", "revenue", "hiring"]:
    results = index.search(question_vectors[key], k=3)
    gap = results[0][0] - results[1][0]
    print(f"Q: {QUESTIONS[key]}")
    for rank, (score, document) in enumerate(results, start=1):
        print(f"   {rank}. {score:.3f}  {chunk_title(document)[:55]}")
    print(f"   gap #1 to #2 = {gap:.3f} {'(confident)' if gap >= 0.1 else '(close call)'}\n")

assert len(index) == 16

**What to notice**

- **Two parallel lists:** `vectors[i]` is the embedding of `documents[i]`. Keeping them at the same position is what links each vector back to its text.
- `search` checks the question against **every** vector. That's called brute-force search, and it's fine for 16 chunks. Real vector databases use approximate shortcuts to stay fast with millions of vectors.
- **The right section isn't always #1.** Summary-style chunks, such as the Executive Summary, mention everything briefly, so they score well on many questions. That's why RAG usually retrieves **more than one** chunk.

## 6. The full RAG flow: retrieve, then answer (paid)

The open-book exam, done properly: look the question up, open the **best few pages**, and answer **only** from those.

The prompt matters as much as the search:

- Put each chunk in its own `<document title="...">` tag, so Claude can say which ones it used.
- Tell Claude to answer **only** from the documents, and what to say if they don't contain the answer. Otherwise it may fill gaps from its own knowledge.

In [ ]:
def build_rag_prompt(question, retrieved):
    documents = "\n\n".join(
        f'<document title="{chunk_title(document).lstrip("# ")}">\n{document}\n</document>'
        for _, document in retrieved
    )
    return f"""Answer the user's question using ONLY the documents below.
If the documents don't contain the answer, say: "The report sections I was given don't say."
Keep the answer short. End with a line "Sources:" listing the document titles you used.

<documents>
{documents}
</documents>

<user_question>
{question}
</user_question>"""


def rag_answer(question, k=2):
    print("STEP 1: embed the question as a 'query'")
    query_vector = generate_embedding(question, input_type="query")

    print(f"STEP 2: search the index and keep the top {k}:")
    retrieved = index.search(query_vector, k=k)
    for score, document in retrieved:
        print(f"           {score:.3f}  {chunk_title(document)}")

    prompt = build_rag_prompt(question, retrieved)
    print(f"STEP 3: build the prompt ({len(prompt):,} characters)")

    response = track(client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
    ))
    print("STEP 4: Claude answers\n")
    print(text_from_message(response))
    return response


rag_response = rag_answer(QUESTIONS["engineering"], k=2)
assert text_from_message(rag_response)

**What to notice**

- Step 1 made no Voyage call: the question's embedding was cached in section 5.
- The two chunks in step 2 are **all** Claude saw. If the answer names Section 2 under *Sources*, you can open that chunk and check the claim. Answers you can trace are one of RAG's big wins.

**RAG against the whole report**, the same question with real token counts:

In [ ]:
whole_report = [(None, report_text)]   # treat the whole report as one retrieved "document"
full_response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": build_rag_prompt(QUESTIONS["engineering"], whole_report)}],
))

rag_tokens = rag_response.usage.input_tokens
full_tokens = full_response.usage.input_tokens
print(f"\nWhole report: {full_tokens:>6,} input tokens")
print(f"RAG (k=2)   : {rag_tokens:>6,} input tokens, {full_tokens / rag_tokens:.1f}x fewer\n")
print("=== Answer from the whole report ===")
print(text_from_message(full_response))

assert rag_tokens < full_tokens

**What to notice**

- On a well-targeted question the two answers should agree. The whole-report answer may add details from other sections, because it sees everything.
- Its *Sources* line can only name the whole report, because we passed it as one document. Small, titled chunks make sources precise.

## 7. Where semantic search fails: an exact ID (paid)

`INC-2023-Q4-011` appears in three sections: 2, 10 and 12. Let's see where semantic search ranks them.

In [ ]:
incident_question = QUESTIONS["incident"]
all_ranked = index.search(question_vectors["incident"], k=len(index))   # rank every section

print(f"Question: {incident_question}\n")
print("Ranks of the sections that contain 'INC-2023-Q4-011':")
for rank, (score, document) in enumerate(all_ranked, start=1):
    if "INC-2023-Q4-011" in document:
        status = "retrieved with k=2" if rank <= 2 else "NOT retrieved with k=2"
        print(f"   rank {rank:>2}/{len(index)}  {score:.3f}  {chunk_title(document)[:45]:<45} {status}")

print("\n--- RAG answer with k=2 ---")
incident_response = rag_answer(incident_question, k=2)

**What to notice**

- An embedding stores **meaning**, not exact strings. To the model, `INC-2023-Q4-011` is just an odd run of characters, so sections are ranked by how much they **sound like** a security incident, not by whether they contain the ID.
- **If one of the three sections ranked below 2,** Claude never saw it, and it can't use what it never sees. Section 12 (Future Outlook) mentions the ID only in passing, so it's the likeliest to be missed.
- Look at how **close** the top scores are. The search isn't confident about anything, which is typical when the key part of a question is an ID, a product code or a name.

**The fix,** in notebook 08: keyword search (BM25), which is good at exact strings, combined with semantic search.

## Final spend

In [ ]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget on Claude.")
print(f"Embeddings saved in {CACHE_PATH}: {len(embedding_cache)}")

## Takeaways

1. **RAG means retrieving a few relevant chunks, then generating** an answer from them. It trades setup work for smaller, cheaper, more focused prompts.
2. **Chunking decides what Claude can see.** Size with overlap is a safe default. Structure-based chunking is cleanest when you control the format.
3. **An embedding is a meaning coordinate.** Similar meaning gives nearby vectors.
4. **Embed chunks as `"document"` and questions as `"query"`.** Batch the calls and cache the results, because embeddings are the part of retrieval that costs money.
5. **Cosine similarity compares directions.** For length-1 vectors it's just a dot product.
6. **Retrieve more than one chunk**, and tell Claude to answer only from them, with sources.
7. **Semantic search misses exact strings** such as IDs and codes. Notebook 08 fixes that.

## Your turn

The first three exercises are free. The last costs about a cent.

- In section 2, try `chunk_by_char(report_text, chunk_size=300)` and `chunk_size=1000`, then re-run the comparison table. How do the mid-word and mixed columns change?
- In section 3, embed the single word `"bug"` and compare it with each section. Which section is it closest to?
- In section 4, add `"slightly off": ([1, 0], [1, 0.2])` to `examples`. Is the score closer to 1 or to 0.7?
- In section 6, ask `rag_answer("Who is the CEO's favourite football team?")`. Does Claude say the sections don't cover it, or does it guess?

**Next:** [08 · RAG advanced](08_rag_advanced.ipynb): BM25 keyword search, hybrid search, reranking and contextual retrieval.